In [45]:
import pandas as pd
import numpy as np
import networkx as nx
import time
from algorithms import spnsa, _single_source_shortest_path_basic, calculate_reliance_from_sources, draw_graph
import random
from tqdm import tqdm

In [46]:
import importlib
import algorithms              # make sure the module itself is imported
importlib.reload(algorithms)   # reloads algorithms.py from disk

from algorithms import spnsa, _single_source_shortest_path_basic, calculate_reliance_from_sources, draw_graph

In [47]:


def create_criminal_graph(file_path:str):
    #G = nx.MultiDiGraph()
    G = nx.DiGraph()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                v = p[2]
                w = p[4]
                G.add_edge(v, w)

    f.close()
    return G

def create_graph(file_path:str):

    #G = nx.MultiDiGraph()
    G = nx.DiGraph()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            v = p[2]
            w = p[4]
            G.add_edge(v, w)

    f.close()
    return G

def get_suspicious_nodes(file_path:str):
    suspicious_nodes = set()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                suspicious_nodes.add(p[2])
                suspicious_nodes.add(p[4])

    f.close()
    return suspicious_nodes

def get_suspicious_edges(file_path:str):
    suspicious_edges = set()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                suspicious_edges.add((p[2], p[4]))

    f.close()
    return suspicious_edges

def find_largest_component(
    G: nx.Graph,
    component_type: str = "weak",  # used only if G is directed
) -> nx.Graph:
    """
    Return the largest component as an induced subgraph (copied).

    - If G is undirected: uses connected_components.
    - If G is directed: uses weakly_connected_components by default,
      or strongly_connected_components if component_type="strong".
    """
    if G.number_of_nodes() == 0:
        # preserve graph type
        return G.copy()

    if G.is_directed():
        if component_type not in {"weak", "strong"}:
            raise ValueError("component_type must be 'weak' or 'strong' for directed graphs.")
        comps = (
            nx.weakly_connected_components(G)
            if component_type == "weak"
            else nx.strongly_connected_components(G)
        )
    else:
        comps = nx.connected_components(G)

    largest_cc = max(comps, key=len)
    return G.subgraph(largest_cc).copy()

def get_connected_components(G):
    if nx.number_connected_components(G) == 1:
        return [G]
        
    connected_components_generatorset = nx.connected_components(G)
    connected_components = []
    for component in connected_components_generatorset:
        component_graph = G.subgraph(component).copy()
        connected_components.append(component_graph)
    return connected_components

def get_eccentricity_distribution_list(component_list):
    ''' component_list : List[component] '''
    eccentricity_distribution_list = []
    for component in component_list:
        eccentricity_distribution = nx.eccentricity(component)
        eccentricity_distribution_list.append(eccentricity_distribution)
    return eccentricity_distribution_list

In [48]:
G_all = create_graph('../../datasets/IBM AML/HI-Medium_Trans.csv')
print(G_all)

DiGraph with 2076999 nodes and 4476959 edges


In [49]:
G_criminal = create_criminal_graph('../../datasets/IBM AML/HI-Medium_Trans.csv')
print(G_criminal)

DiGraph with 41857 nodes and 33199 edges


In [50]:
# largest component H
largest_component = find_largest_component(G_all)
print(largest_component)

DiGraph with 1611244 nodes and 4015935 edges


In [51]:
criminal_nodes = get_suspicious_nodes('../../datasets/IBM AML/HI-Medium_Trans.csv')
criminal_nodes_list = list(criminal_nodes)
all_nodes_list = list(largest_component.nodes())
criminal_edges = get_suspicious_edges('../../datasets/IBM AML/HI-Medium_Trans.csv')

In [52]:
print(len(criminal_nodes))
print(len(criminal_edges))

41857
33199


In [53]:
print(len(criminal_nodes & largest_component.nodes()))

41774


In [9]:
upto = 10
out_degree = dict(largest_component.out_degree())
out_degree_largest = sorted(out_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
out_degree_largest = dict(out_degree_largest)
out_degree_largest

{'100428660': 55956,
 '1004286A8': 35128,
 '1004286F0': 10882,
 '1004289C0': 6840,
 '100428858': 5256,
 '100428810': 5018,
 '1004287C8': 4769,
 '1004288A0': 4537,
 '100428738': 4132,
 '100428978': 4123}

In [10]:
upto = 10
in_degree = dict(largest_component.in_degree())
in_degree_largest = sorted(in_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
in_degree_largest = dict(in_degree_largest)
in_degree_largest

{'100428660': 2304,
 '1004286A8': 1445,
 '1004286F0': 531,
 '1004289C0': 291,
 '100428858': 217,
 '1004288A0': 196,
 '1004287C8': 195,
 '100428810': 187,
 '100428738': 179,
 '100428978': 168}

In [11]:
upto = 10

degree_diff = {
    node: abs(in_degree.get(node, 0) - out_degree.get(node, 0))
    for node in largest_component.nodes()
}

degree_diff_largest = sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:upto]
degree_diff_largest = dict(degree_diff_largest)
degree_diff_largest

{'100428660': 53652,
 '1004286A8': 33683,
 '1004286F0': 10351,
 '1004289C0': 6549,
 '100428858': 5039,
 '100428810': 4831,
 '1004287C8': 4574,
 '1004288A0': 4341,
 '100428978': 3955,
 '100428738': 3953}

In [12]:
len(degree_diff_largest.keys() & in_degree_largest.keys())

10

# Ex0: Uniformly random feed experiments

In [14]:
# feed size 200; radius 1
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []
for i in tqdm(range(100)):
    feed = random.sample(all_nodes_list, 200)
    SPN, paths = spnsa(largest_component, feed, radius=1)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

100%|█████████████████████████████████████████| 100/100 [00:09<00:00, 10.35it/s]

DiGraph with average 376.4 and 176.53 edges
Number of suspicious nodes in SPN: 14.07
Criminal ration in SPN: 0.03738044633368757


In [15]:
# feed size 200; radius 2
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []

for _ in tqdm(range(10)):
    feed = random.sample(all_nodes_list, 200)
    SPN, paths = spnsa(largest_component, feed, radius=2)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

100%|███████████████████████████████████████| 10/10 [3:14:07<00:00, 1164.73s/it]

DiGraph with average 462.0 and 265.3 edges
Number of suspicious nodes in SPN: 21.8
Criminal ration in SPN: 0.047186147186147186


In [16]:
# feed size 500; radius 1
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []

for _ in tqdm(range(100)):
    feed = random.sample(all_nodes_list, 500)
    SPN, paths = spnsa(largest_component, feed, radius=1)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

100%|█████████████████████████████████████████| 100/100 [00:24<00:00,  4.03it/s]

DiGraph with average 941.59 and 442.48 edges
Number of suspicious nodes in SPN: 35.5
Criminal ration in SPN: 0.037702184602640215


# Ex1: Largest degree difference feed experiments

In [18]:
# feed size 200; radius 1
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 262 nodes and 200 edges
Number of suspicious nodes in SPN: 48
Criminal ration in SPN: 0.183206106870229


In [19]:
# feed size 200; radius 2
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 319 nodes and 299 edges
Number of suspicious nodes in SPN: 63
Criminal ration in SPN: 0.1974921630094044


In [20]:
# feed size 500; radius 1
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 610 nodes and 450 edges
Number of suspicious nodes in SPN: 89
Criminal ration in SPN: 0.14590163934426228


# Ex2: Non-zero in-degree & zero out-degree experiments (Collect)

In [21]:
upto = 10

nonzero_in_degree = {
    node: in_degree.get(node, 0)
    for node in largest_component.nodes() if out_degree.get(node, 0) == 0
}

nonzero_in_degree_largest = sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
nonzero_in_degree_largest = dict(nonzero_in_degree_largest)
nonzero_in_degree_largest

{'825369EF0': 95,
 '80ED9B510': 92,
 '84849F1F0': 88,
 '816FB88C0': 84,
 '826683AA0': 83,
 '81DEE6DE0': 83,
 '812FC1710': 82,
 '811C238F0': 79,
 '8229C40C0': 78,
 '81C4CEBF0': 77}

In [22]:
# feed size 200; radius 1
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 200 nodes and 0 edges
Number of suspicious nodes in SPN: 17
Criminal ration in SPN: 0.085


In [23]:
# feed size 200; radius 2
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 200 nodes and 0 edges
Number of suspicious nodes in SPN: 17
Criminal ration in SPN: 0.085


In [24]:
# feed size 500; radius 1
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 500 nodes and 0 edges
Number of suspicious nodes in SPN: 45
Criminal ration in SPN: 0.09


# Ex3: Non-zero out-degree & zero in-degree experiments (Distribute)

In [25]:
upto = 10

nonzero_out_degree = {
    node: out_degree.get(node, 0)
    for node in largest_component.nodes() if in_degree.get(node, 0) == 0
}

nonzero_out_degree_largest = sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
nonzero_out_degree_largest = dict(nonzero_out_degree_largest)
nonzero_out_degree_largest

{'8003D4990': 31,
 '80026B750': 30,
 '8009D89F0': 29,
 '800CA3B60': 28,
 '8019C56F0': 27,
 '83D47DE00': 25,
 '83E923680': 25,
 '800AF6E00': 23,
 '80692D970': 23,
 '805963CD0': 22}

In [26]:
# feed size 200; radius 1
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 517 nodes and 317 edges
Number of suspicious nodes in SPN: 6
Criminal ration in SPN: 0.01160541586073501


In [27]:
# feed size 200; radius 2
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 723 nodes and 524 edges
Number of suspicious nodes in SPN: 31
Criminal ration in SPN: 0.042876901798063624


In [28]:
# feed size 500; radius 1
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 1229 nodes and 765 edges
Number of suspicious nodes in SPN: 40
Criminal ration in SPN: 0.03254678600488202


# Ex4: 50% Collect and 50% Distribute feed experiments

In [29]:
# feed size 200; radius 1
upto = 200
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 358 nodes and 158 edges
Number of suspicious nodes in SPN: 11
Criminal ration in SPN: 0.030726256983240222


In [30]:
# feed size 200; radius 2
upto = 200
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 459 nodes and 259 edges
Number of suspicious nodes in SPN: 28
Criminal ration in SPN: 0.06100217864923747


In [31]:
# feed size 500; radius 1
upto = 500
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 886 nodes and 386 edges
Number of suspicious nodes in SPN: 33
Criminal ration in SPN: 0.03724604966139955


# Ex5: High illicit degrees as feed experiments

In [32]:
upto = 10
illicit_degree = dict(G_criminal.degree())
illicit_degree_largest = sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
illicit_degree_largest = dict(illicit_degree_largest)
illicit_degree_largest

{'100428660': 1488,
 '1004286A8': 944,
 '1004286F0': 277,
 '1004289C0': 189,
 '100428858': 160,
 '1004287C8': 139,
 '100428810': 132,
 '1004288A0': 123,
 '100428780': 113,
 '1004288E8': 112}

In [33]:
# feed size 200; radius 1
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 431 nodes and 564 edges
Number of suspicious nodes in SPN: 338
Criminal ration in SPN: 0.7842227378190255


In [34]:
# feed size 200; radius 2
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 756 nodes and 1100 edges
Number of suspicious nodes in SPN: 590
Criminal ration in SPN: 0.7804232804232805


In [35]:
# feed size 500; radius 1
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 1040 nodes and 1369 edges
Number of suspicious nodes in SPN: 758
Criminal ration in SPN: 0.7288461538461538


In [36]:
# random 200 feed; radius 1
import random

feed = dict(random.sample(list(illicit_degree.items()), 200))
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 399 nodes and 203 edges
Number of suspicious nodes in SPN: 267
Criminal ration in SPN: 0.6691729323308271
